In [1]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [44]:
class Value:
    def __init__(self, data, _children=(),_op='',label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"
    

    def __add__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data + other.data,(self , other), "+")

        def backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = backward

        return out
    
    def __mul__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data,(self,other), "*")

        def backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = backward
        return out

    def __rmul__(self, other):
        return self * other

    def __radd__(self, other):
        return self + other
    
    def __rsub__(self, other):
        return other + (-self)
    
    def __rtruediv__(self, other):
        return other * self **-1
        
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other,(self,), f'**({other})')
        def backward():
            self.grad += other*self.data**(other-1)*out.grad
        out._backward = backward
        return out

    def __truediv__(self,other):
        return self * other ** -1 

    def __neg__(self):
        return -1 * self

    def __sub__(self,other):
        return self + (-other)

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t , (self,), "tanh")
        def backward():
            self.grad += (1 - t**2)*out.grad
        out._backward= backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')
        def backward():
            self.grad += out.data * out.grad
        out._backward = backward

        return out

    def tanh_exp(self):
        x = (2*self).exp()
        out = (x-1)/(x+1)
        return out
    

In [3]:
a = Value(3.0); b = Value(-2.0)
c = a * b + a**2
c.backward()
assert a.grad == 4.0 ,f"a.grad == {a.grad}"
assert b.grad == 3.0 , f"b.grad == {b.grad}"
print("test 1 passed: add and mul gradients")

test 1 passed: add and mul gradients


In [4]:
a = Value(3.0); b = Value(-2.0)
c = a * b + a
c.backward()
assert a.grad == -1.0 ,f"a.grad == {a.grad}"
assert b.grad == 3.0 , f"b.grad == {b.grad}"
print("test 2 passed")

test 2 passed


In [11]:
a = Value(3.0)
c = a + a
c.backward()
assert a.grad == 2.0; f"a.grad == {a.grad}"
print("test 3 passed")

test 3 passed


In [6]:
a = Value (2.0); b = Value(2.0)
c = a ** 2
c.backward()
assert a.grad == 4.0; f"a.grad == {a.grad}"
d = b * b
d.backward()
assert b.grad == a.grad; f"b.grad == {b.grad}"
print("test 4 passed")

test 4 passed


In [7]:
a = Value(2.0); b = Value(-3.0)
c = a * b + b
d = c.tanh()
d.backward()
print(f"a.grad = {a.grad}")


a.grad = -1.8275975133263955e-07


In [9]:
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b = Value(6.8813735870195432)
n = x1*w1 + x2 * w2 + b
o = n.tanh()
o.backward()
print(f"""x1.grad = {x1.grad}, x2.grad = {x2.grad}
            w1.grad = {w1.grad}, w2.grad = {w2.grad},
            b.grad = {b.grad}""")

x1.grad = -1.4999999999999996, x2.grad = 0.4999999999999999
            w1.grad = 0.9999999999999998, w2.grad = 0.0,
            b.grad = 0.4999999999999999


In [10]:
import torch

In [31]:
x1 = torch.tensor([2.0]).double()         ;   x1.requires_grad = True
w1 = torch.tensor([-3.0]).double()        ;   w1.requires_grad = True
x2 = torch.tensor([0.0]).double()         ;   x2.requires_grad = True
w2 = torch.tensor([1.0]).double()         ;   w2.requires_grad = True
b = torch.tensor([6.8813735870195432], dtype= torch.float64)       ;   b.requires_grad = True
n = x1*w1 + x2 * w2 + b
o = n.tanh()
o.backward()

print(f"""
    {x1.grad.item()}, {w1.grad.item()}, {x2.grad.item()}, {w2.grad.item()}, {b.grad.item()}
""")



    -1.4999999999999996, 0.9999999999999998, 0.4999999999999999, 0.0, 0.4999999999999999



In [46]:
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b = Value(6.8813735870195432)
n = x1*w1 + x2 * w2 + b
o = n.tanh_exp()
o.backward()
print(f"""x1.grad = {x1.grad}, x2.grad = {x2.grad}
            w1.grad = {w1.grad}, w2.grad = {w2.grad},
            b.grad = {b.grad}""")

x1.grad = -1.5, x2.grad = 0.5
            w1.grad = 1.0, w2.grad = 0.0,
            b.grad = 0.5


In [72]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self,x):
        act = sum((wi*xi for wi, xi in zip(self.w , x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [73]:
x = [3.0 , 2.0, -1.0]
n = MLP(3,[4,1])
n(x)

Value(data=-0.633713197892254)

In [85]:

xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
n = MLP(3,[4,5,1])
n(x)


Value(data=-0.1381808750322722)

In [86]:
for i in range(50):
    ypred = [n(x) for x in xs]
    loss = sum((ygt - yout)**2 for ygt , yout in zip(ys, ypred))

    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    for p in n.parameters():
        p.data += -0.06*p.grad
    print(i, loss.data)

0 3.608978954190083
1 3.381579415576209
2 3.0933564932051327
3 2.897003647764934
4 2.509257109540191
5 2.2645595176758975
6 1.75822845675747
7 1.4740847208824808
8 0.9231622100976054
9 0.7155416897331972
10 0.3082073351933472
11 0.21561208250862052
12 0.17532424914728295
13 0.14634638454474075
14 0.124653059691333
15 0.10794195321099875
16 0.09475703481614461
17 0.0841413125395224
18 0.07544491121624293
19 0.06821374340541134
20 0.06212248505964815
21 0.056932868220294724
22 0.05246694700135084
23 0.04858949947141403
24 0.045196163204872256
25 0.042205262311338795
26 0.03955206675836216
27 0.03718468835229068
28 0.03506109928354746
29 0.03314693420107381
30 0.03141384800154072
31 0.029838273596505684
32 0.028400471492620455
33 0.02708379495241802
34 0.025874116275267964
35 0.024759374799432365
36 0.023729217786625532
37 0.022774712848723334
38 0.021888115963075855
39 0.02106268303552672
40 0.0202925158415658
41 0.01957243530364025
42 0.018897876653574928
43 0.018264802228913763
44 0.01

In [87]:
ypred

[Value(data=0.9405357157461521),
 Value(data=-0.9776800033573213),
 Value(data=-0.9162867749763522),
 Value(data=0.9358160165910331)]

In [88]:
for i in range(50):
    ypred = [n(x) for x in xs]
    loss = sum((ygt - yout)**2 for ygt , yout in zip(ys, ypred))

    loss.backward()

    for p in n.parameters():
        p.data += -0.06*p.grad
    print(i, loss.data)

0 0.014737440143431594
1 0.013931707280879417
2 0.012821599333677318
3 0.011504540820407574
4 0.010082398343354112
5 0.00864774131962549
6 0.007274698163649833
7 0.006015065018214834
8 0.004898850141639672
9 0.003937693826976721
10 0.0031295542953171497
11 0.0024634244036565436
12 0.0019233494277899259
13 0.0014914565704564266
14 0.0011500026948050538
15 0.0008826000389239228
16 0.0006748289085797478
17 0.0005144351749989519
18 0.0003912726679547555
19 0.000297107227287494
20 0.00022536073279442488
21 0.00017084356536703818
22 0.00012950272916501193
23 9.8198824697531e-05
24 7.451638260748382e-05
25 5.660710029133375e-05
26 4.306297698675431e-05
27 3.281528847684386e-05
28 2.5055151131988936e-05
29 1.9171691694414517e-05
30 1.4704315348203183e-05
31 1.130610530006385e-05
32 8.71591349207628e-06
33 6.73717580708375e-06
34 5.221891568430233e-06
35 4.05854479725326e-06
36 3.1630187923982385e-06
37 2.4717742491057168e-06
38 1.936733180385619e-06
39 1.5214448004491645e-06
40 1.1982127964810

In [89]:
ypred

[Value(data=0.9998956333036643),
 Value(data=-0.9997279257465771),
 Value(data=-0.9997495856653217),
 Value(data=0.9999208591843974)]